#SpendDNA

In [14]:
import pandas as pd
import numpy as np

In [15]:
#Load Dataset
df = pd.read_csv("rahul_transactions.csv")
print("Dataset loaded successfully!")
print("Rows:", len(df))
print("Columns:", len(df.columns))
df.head()

Dataset loaded successfully!
Rows: 1328
Columns: 8


,Date,Time,Description,Type,Amount,Balance,Mode,Ref
0,2024-01-01,03:11,AMAZON SELLER SVCS,Debit,₹2462,678275.0,UPI,TXN190872
1,01-Jan-24,05:44,BHIM-BMTC,DR,50.00,681007.0,UPI,TXN143064
2,01-Jan-24,09:35,NEFT-TECHCRUSH LABS-SALARY MAY24,CR,₹84728,484728.0,NEFT,TXN246316
3,2024-01-01,14:07,UPI-AMAN-8934@OKAXIS,Debit,₹1828,-748745.0,UPI,TXN569226
4,01 Jan 2024,14:23,BHIM-BLINKIT,Debit,270.00,680737.0,UPI,TXN968962


In [16]:
#Getting Raw Dataset before cleaning
print("Dataset Shape:")
print(df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nTransaction Types:")
print(df["Type"].value_counts())

print("\nUnique Descriptions:", df["Description"].nunique())

Dataset Shape:
(1328, 8)

Column Names:
['Date', 'Time', 'Description', 'Type', 'Amount', 'Balance', 'Mode', 'Ref']

Data Types:
Date            object
Time            object
Description     object
Type            object
Amount          object
Balance        float64
Mode            object
Ref             object
dtype: object

Missing Values:
Date           0
Time           0
Description    0
Type           0
Amount         0
Balance        0
Mode           0
Ref            0
dtype: int64

Transaction Types:
Type
DR       670
Debit    652
CR         6
Name: count, dtype: int64

Unique Descriptions: 283


#Feature 1 - Transaction Parser

In [17]:
#Clean Dates
df["date_clean"] = pd.to_datetime(
    df["Date"],
    errors="coerce",
    dayfirst=True,
    format="mixed"
)
print("Date conversion completed.")
print("\nUnparseable dates:")
print(df["date_clean"].isna().sum())
print("\nDate data type:")
print(df["date_clean"].dtype)

Date conversion completed.

Unparseable dates:
0

Date data type:
datetime64[ns]


In [18]:
#Clean Amounts
def clean_amount(value):
    value = str(value)
    value = value.replace("₹", "")
    value = value.replace("Rs.", "")
    value = value.replace(",", "")
    value = value.strip()
    try:
        return float(value)
    except:
        return np.nan

df["amount_clean"] = df["Amount"].apply(clean_amount)
print("Amount conversion completed.")
print("\nUnparseable amounts:")
print(df["amount_clean"].isna().sum())
print("\nAmount data type:")
print(df["amount_clean"].dtype)

Amount conversion completed.

Unparseable amounts:
0

Amount data type:
float64


In [19]:
#Standardise Transacation Type
df["type_clean"] = (
    df["Type"]
    .astype(str)
    .str.strip()
    .str.lower()
)
df["type_clean"] = df["type_clean"].replace({
    "dr": "debit",
    "debit": "debit",
    "cr": "credit",
    "credit": "credit"
})
print(df["type_clean"].value_counts())

type_clean
debit     1322
credit       6
Name: count, dtype: int64


In [20]:
#Remove Duplicates
before = len(df)
df = df.drop_duplicates().copy()
after = len(df)
print("Rows before removing duplicates:", before)
print("Rows after removing duplicates:", after)
print("Duplicates removed:", before - after)

Rows before removing duplicates: 1328
Rows after removing duplicates: 1310
Duplicates removed: 18


In [21]:
#Parser Verification
print("=" * 60)
print("PARSER VERIFICATION")
print("=" * 60)

print("Final rows:", len(df))
print("Duplicate rows:", df.duplicated().sum())
print("Invalid dates:", df["date_clean"].isna().sum())
print("Invalid amounts:", df["amount_clean"].isna().sum())

print("\nCleaned data types:")
print(df[["date_clean", "amount_clean", "type_clean"]].dtypes)

PARSER VERIFICATION
Final rows: 1310
Duplicate rows: 0
Invalid dates: 0
Invalid amounts: 0

Cleaned data types:
date_clean      datetime64[ns]
amount_clean           float64
type_clean              object
dtype: object


#Feature 2 - Vendor Extractor

In [22]:
#Inspect Descriptions
descriptions = df["Description"].dropna().unique()

print("Unique transaction descriptions:")
print(len(descriptions))

for description in descriptions:
    print(description)

Unique transaction descriptions:
283
AMAZON SELLER SVCS
BHIM-BMTC
NEFT-TECHCRUSH LABS-SALARY MAY24
UPI-AMAN-8934@OKAXIS
BHIM-BLINKIT
BHIM ZEPTO
UPI-UBER-2426@HDFCBANK
POS SWIGGY BANGALORE
UPI-GROWWPAY@HDFCBANK
OLA ELECTRIC
BMS MOVIE TICKETS
POS OLA-PRIME
SWIGGY-INSTAMART
UPI-STARBUCKS@AXIS
UPI-THIRDWAVE@OKAXIS
ANI Technologies
BMTC BUS PASS
POS TRUFFLES
FLIPKART INDIA
POS SWIGGY-RESTAURANT
GROFERS INDIA P L
POS UBER BANGALORE
BANGALORE ELEC SUPPLY
TWC INDIA
UPI-BESCOM-BILL@HDFCBANK
UPI-AMAN-0816@OKAXIS
ROPPEN TRANSPORTATION
OLA CABS
POS ZOMATO
UPI-AMAZONPAY@HDFCBANK
POS BLINKIT
IMPS-RENT-LANDLORD-75500265
ZOMATO MEDIA P L
UPI-ANKIT-6430@OKAXIS
UPI-OLACABS@HDFCBANK
UPI-JIORECHARGE@PAYTM
UPI-CCD@HDFCBANK
Swiggy*Order
INSTAMART BANGALORE
UPI-ZOMATO-LIMITED@PAYTM
AVENUE SUPERMARTS
POS HP PETROL STATION
UPI-VIKAS-6060@OKAXIS
POS BANGALORE RESTAURANT
UPI-ZERODHA-COIN@AXIS
BHIM SWIGGY
UPI-BOOKMYSHOW@HDFCBANK
BLINKIT BANGALORE
POS ZEPTO BANGALORE
UPI-DMART@OKAXIS
UPI-RAPIDO@OKAXIS
BUNDL Tech P

In [24]:
#Vendor Keyword Dictionary
vendor_keywords = {

    "Swiggy": ["SWIGGY", "BUNDL"],

    "Zomato": ["ZOMATO"],

    "Zepto": ["ZEPTO"],

    "Amazon": ["AMAZON", "AMZN"],

    "Flipkart": ["FLIPKART"],

    "Myntra": ["MYNTRA"],

    "Uber": ["UBER"],

    "Ola": ["OLA"],

    "Rapido": ["RAPIDO"],

    "Starbucks": ["STARBUCKS"],

    "Netflix": ["NETFLIX"],

    "Spotify": ["SPOTIFY"],

    "Amazon Prime": ["PRIME"],

    "Zerodha": ["ZERODHA"],

    "Groww": ["GROWW"],

    "Cash Withdrawal": ["ATM-WDL"],

    "Personal Transfer": ["UPI-PRIYA"]
}

In [25]:
#Vendor Extraction Function
def extract_vendor(description):

    description = str(description).upper()

    for vendor, keywords in vendor_keywords.items():

        for keyword in keywords:

            if keyword in description:
                return vendor

    return "Uncategorised"


df["vendor_clean"] = df["Description"].apply(extract_vendor)

print("Vendor extraction completed.")

Vendor extraction completed.


In [28]:
#Vendor Verification
print("=" * 60)
print("VENDOR VERIFICATION")
print("=" * 60)

print("Final rows:", len(df))
print("Duplicate rows:", df.duplicated().sum())
print("Invalid dates:", df["date_clean"].isna().sum())
print("Invalid amounts:", df["amount_clean"].isna().sum())
print("Unique canonical vendors:")
print(df["vendor_clean"].nunique())

print("\nTop vendors:")
print(df["vendor_clean"].value_counts().head(10))

VENDOR VERIFICATION
Final rows: 1310
Duplicate rows: 0
Invalid dates: 0
Invalid amounts: 0
Unique canonical vendors:
17

Top vendors:
vendor_clean
Uncategorised    475
Swiggy           223
Zomato           121
Amazon            86
Uber              71
Ola               69
Zepto             58
Flipkart          43
Starbucks         42
Rapido            41
Name: count, dtype: int64


In [29]:
#Vendor CleanUp Audit
uncategorised = df[
    df["vendor_clean"] == "Uncategorised"
]

print("=" * 60)
print("VENDOR CLEANUP AUDIT")
print("=" * 60)

print("Uncategorised transactions:", len(uncategorised))

print("\nDescriptions needing review:")

for description in uncategorised["Description"].unique():
    print(description)

VENDOR CLEANUP AUDIT
Uncategorised transactions: 475

Descriptions needing review:
BHIM-BMTC
NEFT-TECHCRUSH LABS-SALARY MAY24
UPI-AMAN-8934@OKAXIS
BHIM-BLINKIT
BMS MOVIE TICKETS
UPI-THIRDWAVE@OKAXIS
ANI Technologies
BMTC BUS PASS
POS TRUFFLES
GROFERS INDIA P L
BANGALORE ELEC SUPPLY
TWC INDIA
UPI-BESCOM-BILL@HDFCBANK
UPI-AMAN-0816@OKAXIS
ROPPEN TRANSPORTATION
POS BLINKIT
IMPS-RENT-LANDLORD-75500265
UPI-ANKIT-6430@OKAXIS
UPI-JIORECHARGE@PAYTM
UPI-CCD@HDFCBANK
INSTAMART BANGALORE
AVENUE SUPERMARTS
POS HP PETROL STATION
UPI-VIKAS-6060@OKAXIS
POS BANGALORE RESTAURANT
UPI-BOOKMYSHOW@HDFCBANK
BLINKIT BANGALORE
UPI-DMART@OKAXIS
POS CAFE COFFEE DAY
UPI-BWSSB@AXIS
AIRTEL POSTPAID
POS NYKAA BANGALORE
VI POSTPAID
POS BPCL PETROL PUMP
TUMMOC-BMTC
POS INDIAN OIL BANGALORE
POS DINEOUT
UPI-BLINKIT-0656@PAYTM
COFFEE DAY GLOBAL
UPI-NYKAA@AXIS
DISNEY HOTSTAR
POS EMPIRE RESTAURANT
POS MEGHANA FOODS
UPI-IOC9075@HDFCBANK
FKART INTRNET
UPI-BIGBASKET@HDFCBANK
POS DMART BANGALORE
BIGBASKET BANGALORE
POS THIRD 

#Feature 3 - Category Tagger

In [30]:
#Category Mapping
category_map = {

    "Swiggy": "Food Delivery",
    "Zomato": "Food Delivery",

    "Zepto": "Quick Commerce",

    "Amazon": "E-commerce",
    "Flipkart": "E-commerce",
    "Myntra": "E-commerce",

    "Uber": "Transport",
    "Ola": "Transport",
    "Rapido": "Transport",

    "Starbucks": "Cafe",

    "Netflix": "Subscriptions",
    "Spotify": "Subscriptions",
    "Amazon Prime": "Subscriptions",

    "Zerodha": "Investments",
    "Groww": "Investments",

    "Cash Withdrawal": "Cash Withdrawal",
    "Personal Transfer": "Personal Transfer"
}


df["category"] = df["vendor_clean"].map(category_map)

df["category"] = df["category"].fillna("Uncategorised")

print("=" * 60)
print("CATEGORY COUNTS")
print("=" * 60)

print(df["category"].value_counts())

CATEGORY COUNTS
category
Uncategorised        475
Food Delivery        344
Transport            181
E-commerce           149
Quick Commerce        58
Cafe                  42
Investments           23
Subscriptions         18
Cash Withdrawal       17
Personal Transfer      3
Name: count, dtype: int64


#Feature 4 - Spending Overview

In [31]:
#separated credit and debit transactions

credits = df[
    df["type_clean"] == "credit"
].copy()

debits = df[
    df["type_clean"] == "debit"
].copy()

print("Credit transactions:", len(credits))
print("Debit transactions:", len(debits))

Credit transactions: 6
Debit transactions: 1304


In [32]:
#calculated headline financial statistics

total_credits = credits["amount_clean"].sum()

total_debits = debits["amount_clean"].sum()

net_change = total_credits - total_debits

if total_credits != 0:
    savings_rate = (net_change / total_credits) * 100
else:
    savings_rate = 0


print("=" * 60)
print("SPENDING OVERVIEW")
print("=" * 60)

print(f"Total Transactions : {len(df):,}")
print(f"Total Credits      : ₹{total_credits:,.2f}")
print(f"Total Debits       : ₹{total_debits:,.2f}")
print(f"Net Change         : ₹{net_change:,.2f}")
print(f"Savings Rate       : {savings_rate:.2f}%")

SPENDING OVERVIEW
Total Transactions : 1,310
Total Credits      : ₹509,774.00
Total Debits       : ₹1,678,901.00
Net Change         : ₹-1,169,127.00
Savings Rate       : -229.34%


In [33]:
#Spending Data
consumption = debits[
    ~debits["category"].isin([
        "Personal Transfer",
        "Cash Withdrawal"
    ])
].copy()

print("Consumption transactions:", len(consumption))

Consumption transactions: 1284


In [35]:
#Top Categories
top_categories = consumption["category"].value_counts().head(10)

print("=" * 60)
print("TOP CATEGORIES")
print("=" * 60)

print(top_categories)

 #Top 5 Categories
category_spend = (
    consumption
    .groupby("category")["amount_clean"]
    .sum()
    .sort_values(ascending=False)
)

print("=" * 60)
print("TOP 5 CATEGORIES")
print("=" * 60)

for category, amount in category_spend.head(5).items():

    percentage = (
        amount / total_debits
    ) * 100

    print(
        f"{category:<22} "
        f"₹{amount:>12,.2f} "
        f"({percentage:>5.2f}%)"
    )

TOP CATEGORIES
category
Uncategorised     469
Food Delivery     344
Transport         181
E-commerce        149
Quick Commerce     58
Cafe               42
Investments        23
Subscriptions      18
Name: count, dtype: int64
TOP 5 CATEGORIES
E-commerce             ₹  568,804.00 (33.88%)
Uncategorised          ₹  565,331.00 (33.67%)
Investments            ₹  248,160.00 (14.78%)
Food Delivery          ₹  150,839.00 ( 8.98%)
Transport              ₹   49,105.00 ( 2.92%)


In [37]:
#Top Vendors
top_vendors = consumption["vendor_clean"].value_counts().head(10)

print("=" * 60)
print("TOP VENDORS")
print("=" * 60)

print(top_vendors)

#Top 5 Vendors
vendor_spend = (
    consumption
    .groupby("vendor_clean")["amount_clean"]
    .sum()
    .sort_values(ascending=False)
)

print("=" * 60)
print("TOP 5 VENDORS")
print("=" * 60)

for vendor, amount in vendor_spend.head(5).items():

    print(
        f"{vendor:<22} "
        f"₹{amount:>12,.2f}"
    )

TOP VENDORS
vendor_clean
Uncategorised    469
Swiggy           223
Zomato           121
Amazon            86
Uber              71
Ola               69
Zepto             58
Flipkart          43
Starbucks         42
Rapido            41
Name: count, dtype: int64
TOP 5 VENDORS
Uncategorised          ₹  565,331.00
Amazon                 ₹  328,530.00
Zerodha                ₹  210,000.00
Flipkart               ₹  170,745.00
Swiggy                 ₹   95,523.00


#Feature 5 - Monthly analysis

In [39]:
#Month Extraction
consumption["month_number"] = (
    consumption["date_clean"].dt.month
)

month_names = {
    1: "Jan",
    2: "Feb",
    3: "Mar",
    4: "Apr",
    5: "May",
    6: "Jun"
}

consumption["month"] = (
    consumption["month_number"].map(month_names)
)

In [40]:
#Monthly pivot
# AI-assisted: created the category by month spending matrix

month_order = [
    "Jan",
    "Feb",
    "Mar",
    "Apr",
    "May",
    "Jun"
]

monthly_spend = consumption.pivot_table(
    values="amount_clean",
    index="category",
    columns="month",
    aggfunc="sum",
    fill_value=0
)

monthly_spend = monthly_spend.reindex(
    columns=month_order,
    fill_value=0
)

print("=" * 60)
print("MONTHLY SPENDING TREND")
print("=" * 60)

print(monthly_spend.round(2))

MONTHLY SPENDING TREND
month                Jan      Feb       Mar      Apr      May       Jun
category                                                               
Cafe              1029.0   2083.0    2089.0   2727.0   3994.0    3124.0
E-commerce       92182.0  85197.0  103763.0  59583.0  91942.0  136137.0
Food Delivery    22633.0  23740.0   24803.0  27756.0  25408.0   26499.0
Investments      38432.0  15000.0   68644.0  54126.0  48628.0   23330.0
Quick Commerce    3783.0   4321.0    7838.0   3735.0   4643.0    3837.0
Subscriptions      453.0   1036.0    1598.0    538.0    538.0    1341.0
Transport        10111.0   9661.0    5507.0   7163.0  11203.0    5460.0
Uncategorised   118878.0  88033.0  108216.0  89929.0  82086.0   78189.0


In [41]:
#Biggest Growth/Decline
# AI-assisted: compared January and June spending by category

growth_rates = {}

for category in monthly_spend.index:

    january = monthly_spend.loc[category, "Jan"]
    june = monthly_spend.loc[category, "Jun"]

    if january > 0:

        growth = (
            (june - january)
            / january
        ) * 100

        growth_rates[category] = growth


if len(growth_rates) > 0:

    biggest_growth = max(
        growth_rates,
        key=growth_rates.get
    )

    biggest_decline = min(
        growth_rates,
        key=growth_rates.get
    )

    print("Biggest increase:", biggest_growth)
    print(
        "Growth:",
        round(growth_rates[biggest_growth], 2),
        "%"
    )

    print("\nBiggest decline:", biggest_decline)
    print(
        "Change:",
        round(growth_rates[biggest_decline], 2),
        "%"
    )

Biggest increase: Cafe
Growth: 203.6 %

Biggest decline: Transport
Change: -46.0 %


#Feature 6 - Time of day analysis

In [42]:
#Extract hour
consumption["hour"] = (
    consumption["Time"]
    .astype(str)
    .str[:2]
    .astype(int)
)

print(
    consumption["hour"]
    .value_counts()
    .sort_index()
)

hour
0      29
1      27
2      17
3      32
4      34
5      34
6      27
7      31
8      55
9      67
10     63
11     62
12     76
13     62
14     53
15     55
16     65
17     53
18     88
19     84
20    112
21     71
22     50
23     37
Name: count, dtype: int64


In [43]:
#Hourly Spending - AI Assisted
hour_spend = (
    consumption
    .groupby("hour")["amount_clean"]
    .sum()
    .sort_index()
)

print("=" * 60)
print("TIME-OF-DAY SPENDING")
print("=" * 60)

for hour, amount in hour_spend.items():

    bar_length = int(amount / 2000)

    bar = "#" * bar_length

    print(
        f"{hour:02d}:00 | "
        f"₹{amount:>10,.0f} | "
        f"{bar}"
    )

TIME-OF-DAY SPENDING
00:00 | ₹    37,969 | ##################
01:00 | ₹    38,882 | ###################
02:00 | ₹    20,283 | ##########
03:00 | ₹    34,868 | #################
04:00 | ₹    43,901 | #####################
05:00 | ₹    33,671 | ################
06:00 | ₹    55,506 | ###########################
07:00 | ₹    19,938 | #########
08:00 | ₹    37,399 | ##################
09:00 | ₹    76,201 | ######################################
10:00 | ₹   133,526 | ##################################################################
11:00 | ₹   116,389 | ##########################################################
12:00 | ₹    77,096 | ######################################
13:00 | ₹   113,819 | ########################################################
14:00 | ₹   120,010 | ############################################################
15:00 | ₹    39,647 | ###################
16:00 | ₹    95,597 | ###############################################
17:00 | ₹    56,604 | ############################


In [44]:
#Late Night Food Delivery
# AI-assisted: identified late-night Food Delivery transactions

food_delivery = consumption[
    consumption["category"] == "Food Delivery"
].copy()

late_night_food = food_delivery[
    (food_delivery["hour"] >= 21) |
    (food_delivery["hour"] <= 2)
]

if len(food_delivery) > 0:

    late_night_percentage = (
        len(late_night_food)
        / len(food_delivery)
    ) * 100

else:

    late_night_percentage = 0


print("=" * 60)
print("LATE-NIGHT FOOD DELIVERY")
print("=" * 60)

print(
    "Food Delivery transactions:",
    len(food_delivery)
)

print(
    "Late-night transactions:",
    len(late_night_food)
)

print(
    "Late-night percentage:",
    round(late_night_percentage, 2),
    "%"
)

LATE-NIGHT FOOD DELIVERY
Food Delivery transactions: 344
Late-night transactions: 73
Late-night percentage: 21.22 %


#Feature 7 - Anomaly Detection

In [45]:
# AI-assisted: calculated category-wise z-scores for anomaly detection

category_mean = (
    consumption
    .groupby("category")["amount_clean"]
    .transform("mean")
)

category_std = (
    consumption
    .groupby("category")["amount_clean"]
    .transform("std")
)

consumption["z_score"] = (
    (consumption["amount_clean"] - category_mean)
    / category_std
)

anomalies = consumption[
    consumption["z_score"] > 2
].copy()

anomalies = anomalies.sort_values(
    "z_score",
    ascending=False
)

print("=" * 60)
print("ANOMALY DETECTION")
print("=" * 60)

print("Total anomalies:", len(anomalies))

ANOMALY DETECTION
Total anomalies: 38


In [46]:
#Top Anamolies
print(
    anomalies[
        [
            "Date",
            "vendor_clean",
            "category",
            "amount_clean",
            "z_score"
        ]
    ].head(10).to_string(index=False)
)

       Date  vendor_clean      category  amount_clean  z_score
   05/01/24 Uncategorised Uncategorised       18000.0 7.182239
 2024-06-03 Uncategorised Uncategorised       18000.0 7.182239
 2024-05-05 Uncategorised Uncategorised       18000.0 7.182239
   05/04/24 Uncategorised Uncategorised       18000.0 7.182239
   03/03/24 Uncategorised Uncategorised       18000.0 7.182239
  05-Feb-24 Uncategorised Uncategorised       18000.0 7.182239
 2024-03-14 Uncategorised Uncategorised       10505.0 3.976990
  24-Jan-24 Uncategorised Uncategorised       10408.0 3.935508
26 Jun 2024        Amazon    E-commerce       22008.0 3.803781
   07/02/24        Amazon    E-commerce       21986.0 3.799180


#Feature 8 - Spending Personality

In [48]:
#Archetype Functions
# AI-assisted: implemented quantitative spending personality rules


def check_foodie():

    food_categories = [
        "Food Delivery",
        "Restaurants",
        "Cafe"
    ]

    amount = consumption[
        consumption["category"].isin(food_categories)
    ]["amount_clean"].sum()

    percentage = (
        amount / total_debits
    ) * 100

    return percentage > 25


def check_quick_commerce():

    amount = consumption[
        consumption["category"] == "Quick Commerce"
    ]["amount_clean"].sum()

    percentage = (
        amount / total_debits
    ) * 100

    return percentage > 15


def check_shopaholic():

    amount = consumption[
        consumption["category"] == "E-commerce"
    ]["amount_clean"].sum()

    percentage = (
        amount / total_debits
    ) * 100

    return percentage > 15


def check_investor():

    amount = consumption[
        consumption["category"] == "Investments"
    ]["amount_clean"].sum()

    percentage = (
        amount / total_debits
    ) * 100

    return percentage > 15


def check_late_night_snacker():

    return late_night_percentage > 50


def check_cab_commuter():

    amount = consumption[
        consumption["category"] == "Transport"
    ]["amount_clean"].sum()

    percentage = (
        amount / total_debits
    ) * 100

    return percentage > 10


def check_subscription_lover():

    vendor_count = consumption[
        consumption["category"] == "Subscriptions"
    ]["vendor_clean"].nunique()

    return vendor_count >= 5


def check_yolo_spender():

    return savings_rate < 10


def check_disciplined_saver():

    return savings_rate > 40

In [50]:
#Apply Archetypes
archetypes = []

if check_foodie():
    archetypes.append("THE FOODIE")

if check_quick_commerce():
    archetypes.append("THE QUICK COMMERCE JUNKIE")

if check_shopaholic():
    archetypes.append("THE SHOPAHOLIC")

if check_investor():
    archetypes.append("THE INVESTOR")

if check_late_night_snacker():
    archetypes.append("THE LATE-NIGHT SNACKER")

if check_cab_commuter():
    archetypes.append("THE CAB COMMUTER")

if check_subscription_lover():
    archetypes.append("THE SUBSCRIPTION LOVER")

if check_yolo_spender():
    archetypes.append("THE YOLO SPENDER")

if check_disciplined_saver():
    archetypes.append("THE DISCIPLINED SAVER")


print("=" * 60)
print("SPENDING PERSONALITY")
print("=" * 60)

for archetype in archetypes:
    print("★", archetype)



SPENDING PERSONALITY
★ THE SHOPAHOLIC
★ THE YOLO SPENDER


In [52]:
#Invented Archetyoe
cafe_vendors = consumption[
    consumption["category"] == "Cafe"
]["vendor_clean"].nunique()

if cafe_vendors >= 2:

    print("\n★ THE PAVEMENT COFFEE CONNOISSEUR")
    print(
        "Reason:",
        cafe_vendors,
        "different cafe vendors"
    )

In [53]:
#Day of week analysis
consumption["day_of_week"] = (
    consumption["date_clean"].dt.day_name()
)

weekday_names = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday"
]

weekend_names = [
    "Saturday",
    "Sunday"
]


weekday_total = consumption[
    consumption["day_of_week"].isin(weekday_names)
]["amount_clean"].sum()

weekend_total = consumption[
    consumption["day_of_week"].isin(weekend_names)
]["amount_clean"].sum()


average_weekday = weekday_total / 5
average_weekend = weekend_total / 2


if average_weekday > 0:

    weekend_difference = (
        (average_weekend - average_weekday)
        / average_weekday
    ) * 100

else:

    weekend_difference = 0


print("=" * 60)
print("DAY-OF-WEEK ANALYSIS")
print("=" * 60)

print(
    f"Average weekday spending : ₹{average_weekday:,.2f}"
)

print(
    f"Average weekend spending : ₹{average_weekend:,.2f}"
)

print(
    f"Weekend difference       : {weekend_difference:.2f}%"
)

if weekend_difference >= 50:
    print("Weekend spending is 50% or more than weekday spending.")
else:
    print("Weekend spending is less than 50% above weekday spending.")

DAY-OF-WEEK ANALYSIS
Average weekday spending : ₹233,155.40
Average weekend spending : ₹232,584.50
Weekend difference       : -0.24%
Weekend spending is less than 50% above weekday spending.


In [54]:
#Spend Forecasting
forecast = {}

for category in monthly_spend.index:

    last_three_months = monthly_spend.loc[
        category,
        ["Apr", "May", "Jun"]
    ].values.astype(float)

    projected_amount = np.mean(
        last_three_months
    )

    forecast[category] = projected_amount


print("=" * 60)
print("NEXT MONTH SPENDING FORECAST")
print("=" * 60)

for category, amount in forecast.items():

    print(
        f"{category:<25} "
        f"₹{amount:>12,.2f}"
    )

NEXT MONTH SPENDING FORECAST
Cafe                      ₹    3,281.67
E-commerce                ₹   95,887.33
Food Delivery             ₹   26,554.33
Investments               ₹   42,028.00
Quick Commerce            ₹    4,071.67
Subscriptions             ₹      805.67
Transport                 ₹    7,942.00
Uncategorised             ₹   83,401.33


#Feature 9 - Final Report

In [57]:
print("\n")
print("=" * 70)
print("                  SPEND-DNA REPORT")
print("                  RAHUL SHARMA")
print("=" * 70)

print("\nEXECUTIVE SUMMARY")
print("-" * 70)

print(f"Transactions       : {len(df):,}")
print(f"Unique Vendors     : {df['vendor_clean'].nunique()}")
print(f"Total Credits      : ₹{total_credits:,.2f}")
print(f"Total Debits       : ₹{total_debits:,.2f}")
print(f"Net Change         : ₹{net_change:,.2f}")
print(f"Savings Rate       : {savings_rate:.2f}%")


print("\nTOP CATEGORIES")
print("-" * 70)

for category, amount in category_spend.head(5).items():

    percentage = (
        amount / total_debits
    ) * 100

    print(
        f"{category:<25}"
        f"{percentage:>7.2f}%   "
        f"₹{amount:>12,.2f}"
    )


print("\nTOP VENDORS")
print("-" * 70)

for vendor, amount in vendor_spend.head(5).items():

    print(
        f"{vendor:<25}"
        f"₹{amount:>12,.2f}"
    )


print("\nTIME-OF-DAY")
print("-" * 70)

most_active_hour = (
    hour_spend.idxmax()
)

print(
    f"Highest spending hour : "
    f"{most_active_hour:02d}:00"
)

print(
    f"Late-night Food Delivery : "
    f"{late_night_percentage:.2f}%"
)


print("\nANOMALIES")
print("-" * 70)

print(
    "Anomalies detected:",
    len(anomalies)
)


print("\nSPENDING PERSONALITY")
print("-" * 70)

for archetype in archetypes:

    print("★", archetype)


print("\n" + "=" * 70)
print("                 SPENDDNA COMPLETE")
print("=" * 70)



                  SPEND-DNA REPORT
                  RAHUL SHARMA

EXECUTIVE SUMMARY
----------------------------------------------------------------------
Transactions       : 1,310
Unique Vendors     : 17
Total Credits      : ₹509,774.00
Total Debits       : ₹1,678,901.00
Net Change         : ₹-1,169,127.00
Savings Rate       : -229.34%

TOP CATEGORIES
----------------------------------------------------------------------
E-commerce                 33.88%   ₹  568,804.00
Uncategorised              33.67%   ₹  565,331.00
Investments                14.78%   ₹  248,160.00
Food Delivery               8.98%   ₹  150,839.00
Transport                   2.92%   ₹   49,105.00

TOP VENDORS
----------------------------------------------------------------------
Uncategorised            ₹  565,331.00
Amazon                   ₹  328,530.00
Zerodha                  ₹  210,000.00
Flipkart                 ₹  170,745.00
Swiggy                   ₹   95,523.00

TIME-OF-DAY
----------------------------

Three Data-Specific Insights

# Key Insights

### Insight 1
Food Delivery was one of the major spending categories during the six-month period.

### Insight 2
A significant portion of Food Delivery transactions occurred during late-night hours.

### Insight 3
The transaction analysis identified several unusually large transactions within their respective spending categories.